**13/08/2026** -- Inline descriptive statistics for results section *Deprivation*


In [1]:
library(dplyr)
library(readr)
library(tidyr)
library(lme4)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack




In [2]:
root <- rprojroot::find_root(rprojroot::has_file(".gitignore"))
source(file.path(root, "src/deprivation_pm_bhm/data_prep.R"))
source(file.path(root, "src/deprivation_pm_bhm/pca.R"))

In [3]:
SCRATCH_DIR <- Sys.getenv("SCRATCH_DIR")
DATA_PATH   <- file.path(SCRATCH_DIR, 
                         "data/spatial/fire_pm_dep_paper_data",
                         "proc_data/df_af_annual_2000_2023.csv")

In [4]:
df <- read_csv(DATA_PATH)

Rows: 994412 Columns: 74
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (11): geometry, ISO_A3, country, region, continent, INCOME_GRP, ECONOMY,...
dbl (63): lon, lat, total_PM25, total_O3, fire_PM25, fire_O3, fire_PM25_hu, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
# Filter to <= 2022 (GDP data only available to 2022)
df <- df |> 
    filter(year <= 2022) |> 
    group_by(lon, lat) |>
    mutate(grid_id = cur_group_id()) |> # create grid_id for projecting SE vars
    ungroup()

##### Estimate PPCA deprivation index for 2000-2017:

In [7]:
# Do PCA, keep augmented dataset with PCs
df_pca_17  <- compute_pca( 
    df |> filter(year <= 2017),
    method          = "ppca",
    se_indicators   = c("stunting_pct_u5", "child_dep_pct", "edu_mean_years", "log_GDP_pc", "imp_san_access_pct"),
    n_pcs           = 5,
    scale           = TRUE,
    centre          = TRUE,
    seed            = 42,
    positive_vars   = c("stunting_pct_u5", "child_dep_pct"),
    negative_vars   = c("edu_mean_years", "log_GDP_pc", "imp_san_access_pct")
)$data

# Join PCs back onto df (helper funct)
df_17  <- prepare_analysis_data(df |> filter(year <= 2017), 
                             df_pca_17, 
                             outcome = "fire_PM25_hu", 
                             scale_y = FALSE,
                             scale_PC1 = TRUE)

Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(se_indicators)

  # Now:
  data %>% select(all_of(se_indicators))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”


##### Impute socioeconomic indicators (education, sanitation, stunting from 2018-2022) and estimate PPCA for 2000-2022:

In [11]:
df_22 <- project_indicators(
    df,
    list(edu_mean_years     = 2017,
         stunting_pct_u5    = 2017,
         imp_san_access_pct = 2017)
)

[1] "Projecting edu_mean_years forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0227167 (tol = 0.002, component 1)”
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”


[1] "Projecting stunting_pct_u5 forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0278442 (tol = 0.002, component 1)”
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”


[1] "Projecting imp_san_access_pct forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.00349425 (tol = 0.002, component 1)”


In [12]:
# Do PCA
df_pca_22  <- compute_pca( 
    df |> filter(year <= 2022),
    method          = "ppca",
    se_indicators   = c("stunting_pct_u5", "child_dep_pct", "edu_mean_years", "log_GDP_pc", "imp_san_access_pct"),
    n_pcs           = 5,
    scale           = TRUE,
    centre          = TRUE,
    seed            = 42,
    positive_vars   = c("stunting_pct_u5", "child_dep_pct"),
    negative_vars   = c("edu_mean_years", "log_GDP_pc", "imp_san_access_pct")
)$data

# Join PCs back onto df (helper funct)
df_22  <- prepare_analysis_data(df_22 |> filter(year <= 2022), 
                             df_pca_22, 
                             outcome = "fire_PM25_hu", 
                             scale_y = FALSE,
                             scale_PC1 = TRUE)

Warning message in orth(C):
“Precision for components 4 - 5 is below .Machine$double.eps. 
Results for those components are likely to be inaccurate!!
”


##### “The five countries with the highest average deprivation levels were Niger, Chad, Ethiopia, Burundi and South Sudan ... those with the lowest levels were Tunisia, Algeria, Libya, South Africa, and Egypt”

2000-2017

In [52]:
pw_avg_PC1_country <- df_17 %>% 
    mutate(
        pw_PC1 = PC1 * pop_count,
        region = recode(region, "Middle Africa" = "Central Africa")
    ) %>% 
    group_by(country, year) %>% 
    summarise(
        region = first(region),
        across( c("pw_PC1", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    mutate(
        pw_avg_PC1 = pw_PC1 / pop_count
    ) %>% 
    group_by(country) %>% 
    summarise(
        region = first(region),
        pw_avg_PC1 = mean(pw_avg_PC1)
    ) |> 
    arrange(desc(pw_avg_PC1))

In [53]:
pw_avg_PC1_country |> head(5)

country,region,pw_avg_PC1
<chr>,<chr>,<dbl>
Niger,Western Africa,1.3078861
Chad,Central Africa,0.9791120
Ethiopia,Eastern Africa,0.8709453
Burundi,Eastern Africa,0.6920492
South Sudan,Eastern Africa,0.6478516


In [54]:
pw_avg_PC1_country |> tail(5)

country,region,pw_avg_PC1
<chr>,<chr>,<dbl>
"Egypt, Arab Republic of",Northern Africa,-1.756751
South Africa,Southern Africa,-1.848522
Libya,Northern Africa,-1.993051
Algeria,Northern Africa,-2.038365
Tunisia,Northern Africa,-2.186216


2000-2022 (with imputation):

In [55]:
pw_avg_PC1_country <- df_22 %>% 
    mutate(
        pw_PC1 = PC1 * pop_count,
        region = recode(region, "Middle Africa" = "Central Africa")
    ) %>% 
    group_by(country, year) %>% 
    summarise(
        region = first(region),
        across( c("pw_PC1", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    mutate(
        pw_avg_PC1 = pw_PC1 / pop_count
    ) %>% 
    group_by(country) %>% 
    summarise(
        region = first(region),
        pw_avg_PC1 = mean(pw_avg_PC1)
    ) |> 
    arrange(desc(pw_avg_PC1))

In [56]:
pw_avg_PC1_country |> head(5)

country,region,pw_avg_PC1
<chr>,<chr>,<dbl>
Niger,Western Africa,1.2479487
Chad,Central Africa,0.9563576
Ethiopia,Eastern Africa,0.7353965
South Sudan,Eastern Africa,0.7242580
Burundi,Eastern Africa,0.6993951


In [57]:
pw_avg_PC1_country |> tail(5)

country,region,pw_avg_PC1
<chr>,<chr>,<dbl>
"Egypt, Arab Republic of",Northern Africa,-1.729507
South Africa,Southern Africa,-1.819426
Libya,Northern Africa,-1.884286
Algeria,Northern Africa,-1.966766
Tunisia,Northern Africa,-2.117851


##### “Rural areas were systematically characterised by higher deprivation scores than urban areas in all countries except Cabo Verde”

2000-2017:

In [18]:
pw_avg_PC1_country_ur <- df_17 %>% 
    filter(!is.na(urban_rural_cat)) %>% 
    mutate(
        pw_PC1 = PC1 * pop_count,
        region = recode(region, "Middle Africa" = "Central Africa")
    ) %>% 
    group_by(country, urban_rural_cat, year) %>% 
    summarise(
        region = first(region),
        across( c("pw_PC1", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    mutate(
        pw_avg_PC1 = pw_PC1 / pop_count
    ) %>% 
    group_by(country, urban_rural_cat) %>% 
    summarise(
        region = first(region),
        pw_avg_PC1 = mean(pw_avg_PC1),
        .groups = "drop"
    ) %>%
    tidyr::complete(
        country,
        urban_rural_cat = c("urban", "rural")
    ) %>% 
    group_by(country) %>%
    tidyr::fill(region, .direction = "downup") %>%
    ungroup()

In [21]:
# Pivot to wide form
pw_avg_PC1_country_ur_wide <- pw_avg_PC1_country_ur %>% 
    pivot_wider(
        id_cols = c("country", "region"),
        names_from = "urban_rural_cat",
        values_from = "pw_avg_PC1",
        names_prefix = "pw_avg_PC1_"
    ) %>% 
    mutate(rural_gt_urban = pw_avg_PC1_rural > pw_avg_PC1_urban)

In [23]:
pw_avg_PC1_country_ur_wide |> count(rural_gt_urban)

rural_gt_urban,n
<lgl>,<int>
FALSE,1
TRUE,47
NA,4


In [25]:
# One country where rural deprivation is lower than urban
pw_avg_PC1_country_ur_wide |> filter(!rural_gt_urban)

country,region,pw_avg_PC1_rural,pw_avg_PC1_urban,rural_gt_urban
<chr>,<chr>,<dbl>,<dbl>,<lgl>
Cabo Verde,Western Africa,-1.029098,-0.8471759,FALSE


2000-2022:

In [26]:
pw_avg_PC1_country_ur <- df_22 %>% 
    filter(!is.na(urban_rural_cat)) %>% 
    mutate(
        pw_PC1 = PC1 * pop_count,
        region = recode(region, "Middle Africa" = "Central Africa")
    ) %>% 
    group_by(country, urban_rural_cat, year) %>% 
    summarise(
        region = first(region),
        across( c("pw_PC1", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    mutate(
        pw_avg_PC1 = pw_PC1 / pop_count
    ) %>% 
    group_by(country, urban_rural_cat) %>% 
    summarise(
        region = first(region),
        pw_avg_PC1 = mean(pw_avg_PC1),
        .groups = "drop"
    ) %>%
    tidyr::complete(
        country,
        urban_rural_cat = c("urban", "rural")
    ) %>% 
    group_by(country) %>%
    tidyr::fill(region, .direction = "downup") %>%
    ungroup()

In [27]:
# Pivot to wide form
pw_avg_PC1_country_ur_wide <- pw_avg_PC1_country_ur %>% 
    pivot_wider(
        id_cols = c("country", "region"),
        names_from = "urban_rural_cat",
        values_from = "pw_avg_PC1",
        names_prefix = "pw_avg_PC1_"
    ) %>% 
    mutate(rural_gt_urban = pw_avg_PC1_rural > pw_avg_PC1_urban)

In [28]:
pw_avg_PC1_country_ur_wide |> count(rural_gt_urban)

rural_gt_urban,n
<lgl>,<int>
FALSE,1
TRUE,51


In [29]:
# One country where rural deprivation is lower than urban
pw_avg_PC1_country_ur_wide |> filter(!rural_gt_urban)

country,region,pw_avg_PC1_rural,pw_avg_PC1_urban,rural_gt_urban
<chr>,<chr>,<dbl>,<dbl>,<lgl>
Cabo Verde,Western Africa,-1.092867,-0.9316057,FALSE


##### “Both variables had large between-country heterogeneity: approximately X% of the variance in each was attributed to differences between countries (intraclass correlation coefficients (ICC) were ICCfire = 0.X, ICCdeprivation = 0.Y for fire PM2.5 and deprivation respectively)”
Note that ICC fire PM2.5 is shown here also -- because this notebook estimates PPCA deprivation indicies using the original (2000-2017) and imputed (2000-2022) socioeconomic data. So may as well do this result here to avoid repeating 2x PPCA in another notebook

In [33]:
# Function to calculate ICC
calc_icc <- function(model) {
    var_components <- as.data.frame(VarCorr(model))
    between <- var_components$vcov[var_components$grp == "country"]
    within <- var_components$vcov[var_components$grp == "Residual"]
    return(between / (between + within))
}

Compute ICC by year, then average:

2000-2017:

In [37]:
years_sel <- 2000:2017

# Calculate ICC for each year
icc_results <- data.frame(
    year = years_sel,
    icc_PM = NA_real_,
    icc_PC1 = NA_real_
)

for (i in seq_along(years_sel)) {
    year_data <- df_17 %>% filter(year == years_sel[i])
    
    # Fire PM2.5
    fit_PM <- lmer(fire_PM25_hu ~ 1 + (1|country), data = year_data, REML = TRUE)
    icc_results$icc_PM[i] <- calc_icc(fit_PM)
    
    # Deprivation (PC1)
    fit_PC1 <- lmer(PC1 ~ 1 + (1|country), data = year_data, REML = TRUE)
    icc_results$icc_PC1[i] <- calc_icc(fit_PC1)
}

In [41]:
# Summary statistics
cat("=== ICC Summary", years_sel[1], "-", years_sel[length(years_sel)], "===\n\n")

cat("Fire PM2.5:\n")
cat("  Mean ICC:", round(mean(icc_results$icc_PM), 3), "\n")
cat("  SD:", round(sd(icc_results$icc_PM), 4), "\n")
cat("  Range:", round(min(icc_results$icc_PM), 3), "-", 
    round(max(icc_results$icc_PM), 3), "\n\n")

cat("Deprivation:\n")
cat("  Mean ICC:", round(mean(icc_results$icc_PC1), 3), "\n")
cat("  SD:", round(sd(icc_results$icc_PC1), 4), "\n")
cat("  Range:", round(min(icc_results$icc_PC1), 3), "-", 
    round(max(icc_results$icc_PC1), 3), "\n\n")

# Within-country shares
within_share_PM <- 1 - mean(icc_results$icc_PM)
within_share_PC1 <- 1 - mean(icc_results$icc_PC1)

cat("Within-country variance share:\n")
cat("  Fire PM2.5:", round(within_share_PM, 3), 
    "(", round(within_share_PM * 100, 1), "%)\n")
cat("  Deprivation:", round(within_share_PC1, 3), 
    "(", round(within_share_PC1 * 100, 1), "%)\n\n")

=== ICC Summary 2000 - 2017 ===

Fire PM2.5:
  Mean ICC: 0.738 
  SD: 0.0477 
  Range: 0.609 - 0.797 

Deprivation:
  Mean ICC: 0.816 
  SD: 0.0141 
  Range: 0.791 - 0.835 

Within-country variance share:
  Fire PM2.5: 0.262 ( 26.2 %)
  Deprivation: 0.184 ( 18.4 %)



2000-2022:

In [42]:
years_sel <- 2000:2022

# Calculate ICC for each year
icc_results <- data.frame(
    year = years_sel,
    icc_PM = NA_real_,
    icc_PC1 = NA_real_
)

for (i in seq_along(years_sel)) {
    year_data <- df_22 %>% filter(year == years_sel[i])
    
    # Fire PM2.5
    fit_PM <- lmer(fire_PM25_hu ~ 1 + (1|country), data = year_data, REML = TRUE)
    icc_results$icc_PM[i] <- calc_icc(fit_PM)
    
    # Deprivation (PC1)
    fit_PC1 <- lmer(PC1 ~ 1 + (1|country), data = year_data, REML = TRUE)
    icc_results$icc_PC1[i] <- calc_icc(fit_PC1)
}

In [43]:
# Summary statistics
cat("=== ICC Summary", years_sel[1], "-", years_sel[length(years_sel)], "===\n\n")

cat("Fire PM2.5:\n")
cat("  Mean ICC:", round(mean(icc_results$icc_PM), 3), "\n")
cat("  SD:", round(sd(icc_results$icc_PM), 4), "\n")
cat("  Range:", round(min(icc_results$icc_PM), 3), "-", 
    round(max(icc_results$icc_PM), 3), "\n\n")

cat("Deprivation:\n")
cat("  Mean ICC:", round(mean(icc_results$icc_PC1), 3), "\n")
cat("  SD:", round(sd(icc_results$icc_PC1), 4), "\n")
cat("  Range:", round(min(icc_results$icc_PC1), 3), "-", 
    round(max(icc_results$icc_PC1), 3), "\n\n")

# Within-country shares
within_share_PM <- 1 - mean(icc_results$icc_PM)
within_share_PC1 <- 1 - mean(icc_results$icc_PC1)

cat("Within-country variance share:\n")
cat("  Fire PM2.5:", round(within_share_PM, 3), 
    "(", round(within_share_PM * 100, 1), "%)\n")
cat("  Deprivation:", round(within_share_PC1, 3), 
    "(", round(within_share_PC1 * 100, 1), "%)\n\n")

=== ICC Summary 2000 - 2022 ===

Fire PM2.5:
  Mean ICC: 0.731 
  SD: 0.0555 
  Range: 0.585 - 0.797 

Deprivation:
  Mean ICC: 0.822 
  SD: 0.0178 
  Range: 0.793 - 0.85 

Within-country variance share:
  Fire PM2.5: 0.269 ( 26.9 %)
  Deprivation: 0.178 ( 17.8 %)

